# 02 · Preprocesamiento sin fuga de datos

Pasos:
1. Leer CSV (`na_values="?"`) → 2. primer encuentro por paciente → 3. quitar fallecidos/hospicio
→ 4. target binario → 5. quitar columnas inútiles → 6. agrupar ICD-9 / edad / ids
→ 7. **split estratificado 80/20** → 8. `Pipeline` (imputación + one-hot) **ajustado solo con train**.

La lógica vive en `src/data.py` y `src/features.py`; aquí se ejecuta y se **valida**.

In [1]:
import numpy as np
import pandas as pd

from src.data import build_xy, filter_encounters, load_raw, split_train_test
from src.features import SPECIALTY, build_preprocessor, engineer, icd9_to_group

pd.set_option("display.width", 140)

## 1-5. Carga, filtros, target y columnas

In [2]:
raw = load_raw()
filtered = filter_encounters(raw)
X, y = build_xy(filtered)

print(
    f"CSV crudo:            {len(raw):>7,} encuentros / {raw['patient_nbr'].nunique():,} pacientes"
)
print(f"Tras los filtros:     {len(filtered):>7,} filas")
print(f"patient_nbr únicos:   {filtered['patient_nbr'].is_unique}")
print(
    f"Fallecimiento/hospicio restantes: "
    f"{filtered['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21]).sum()}"
)
print(f"Positivos (<30):      {y.sum():,} ({y.mean():.2%})")
print(
    f"Columnas: {raw.shape[1]} → {X.shape[1]}  (fuera: encounter_id, patient_nbr, weight, payer_code, "
    "examide, citoglipton, readmitted)"
)

CSV crudo:            101,766 encuentros / 71,518 pacientes
Tras los filtros:      69,973 filas
patient_nbr únicos:   True
Fallecimiento/hospicio restantes: 0
Positivos (<30):      6,277 (8.97%)
Columnas: 50 → 43  (fuera: encounter_id, patient_nbr, weight, payer_code, examide, citoglipton, readmitted)


## 6. Variables deterministas (`engineer`)
Mapeo ICD-9 → grupo clínico (Strack 2014). Los prefijos `V`/`E` van a `Other` y NaN a `Missing`.

In [3]:
for code in ["250.83", "428", "786", "715", "V27", "E909", np.nan]:
    print(f"{str(code):>7} → {icd9_to_group(code)}")

 250.83 → Diabetes
    428 → Circulatory
    786 → Respiratory
    715 → Musculoskeletal
    V27 → Other
   E909 → Other
    nan → Missing


In [4]:
X = engineer(X)
print(X.shape)
print(X["diag_1_group"].value_counts().to_string())

(69973, 43)
diag_1_group
Circulatory        21384
Other              12122
Respiratory         9486
Digestive           6487
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3440
Neoplasms           2538
Missing               10


`engineer` no aprende nada de los datos (es una función fija), así que aplicarlo antes del split
**no** produce fuga. Convierte además `age` a punto medio numérico y los tres `*_id` a texto
(son categorías, no cantidades).

## 7. Split estratificado 80/20 (antes de imputar / codificar)

In [5]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
print(f"train: {X_train.shape}  positivos {y_train.mean():.2%}")
print(f"test:  {X_test.shape}  positivos {y_test.mean():.2%}")
assert abs(y_train.mean() - y_test.mean()) < 0.002  # estratificación

train: (55978, 43)  positivos 8.97%
test:  (13995, 43)  positivos 8.97%


## 8. Preprocesador (imputación + one-hot), ajustado SOLO con train
Faltantes que quedan tras `engineer`: `race` (moda) y `medical_specialty` (categoría `Unknown`).

In [6]:
print(X_train.isna().sum()[lambda s: s > 0].to_string())

prep = build_preprocessor().fit(X_train)
names = prep.get_feature_names_out()
Xt_train = prep.transform(X_train)
Xt_test = prep.transform(X_test)
print("\nMatriz transformada:", Xt_train.shape, "→", len(names), "variables")
print(
    "NaN tras transformar (train/test):",
    int(np.isnan(Xt_train).sum()),
    int(np.isnan(Xt_test).sum()),
)

race                  1554
medical_specialty    26972

Matriz transformada: (55978, 182) → 182 variables
NaN tras transformar (train/test): 0 0


### Comprobación anti-fuga
La moda de `race` y las categorías de `medical_specialty` deben salir de **train**.

In [7]:
race_mode_train = X_train["race"].mode()[0]
imp = prep.named_transformers_["cat"].named_steps["impute"]
cat_cols = prep.named_transformers_["cat"].feature_names_in_.tolist()
learned = imp.statistics_[cat_cols.index("race")]
print("moda de race aprendida por el Pipeline:", learned, "| moda en train:", race_mode_train)
assert learned == race_mode_train

spec_cats = [c for c in names if c.startswith(SPECIALTY)]
print(f"\n{len(spec_cats)} columnas de medical_specialty (top-10 + Unknown + Otra):")
print(spec_cats)

moda de race aprendida por el Pipeline: Caucasian | moda en train: Caucasian

12 columnas de medical_specialty (top-10 + Unknown + Otra):
['medical_specialty_Cardiology', 'medical_specialty_Emergency/Trauma', 'medical_specialty_Family/GeneralPractice', 'medical_specialty_InternalMedicine', 'medical_specialty_Nephrology', 'medical_specialty_Orthopedics', 'medical_specialty_Orthopedics-Reconstructive', 'medical_specialty_Pulmonology', 'medical_specialty_Radiologist', 'medical_specialty_Surgery-General', 'medical_specialty_Unknown', 'medical_specialty_infrequent_sklearn']


## 9. Escalado de variables numéricas
**Cómo:** `StandardScaler` (estandarización z-score): `z = (x - μ) / σ`, con μ y σ calculadas **solo con train**
y aplicadas igual a test. Va dentro del `Pipeline` numérico, después de la imputación por mediana
(`SimpleImputer → StandardScaler`), y solo afecta a las columnas numéricas; las columnas one-hot (0/1)
no se escalan.

**Por qué:**
- Las variables numéricas tienen rangos muy distintos (p. ej. `time_in_hospital` ~1-14 frente a
  `num_lab_procedures` ~1-130 o `number_inpatient` ~0-20). Sin escalar, las de mayor magnitud dominan
  la penalización de la regresión logística (L2) y la distancia del kernel RBF de la SVM, y la
  optimización converge peor.
- **Solo se escala para modelos sensibles a la escala** (`build_preprocessor(scale=True)`: regresión
  logística y SVM). Los modelos basados en árboles (Random Forest, boosting) son invariantes a
  transformaciones monótonas, así que escalar no aporta nada y se evita.
- Se elige z-score y no min-max porque los conteos tienen outliers y min-max comprimiría el resto de
  valores en un intervalo muy pequeño.
- Ajustar μ y σ solo con train evita fuga de datos; dentro de la validación cruzada se re-ajustan en cada fold.


In [8]:
prep_scaled = build_preprocessor(scale=True).fit(X_train)
num_cols = prep_scaled.named_transformers_["num"].feature_names_in_.tolist()
Xs_train = pd.DataFrame(prep_scaled.transform(X_train), columns=prep_scaled.get_feature_names_out())
Xs_test = pd.DataFrame(prep_scaled.transform(X_test), columns=prep_scaled.get_feature_names_out())

summary = pd.DataFrame(
    {
        "rango_original": [f"{X_train[c].min():g} – {X_train[c].max():g}" for c in num_cols],
        "media_train": Xs_train[num_cols].mean().round(3).values,
        "std_train": Xs_train[num_cols].std(ddof=0).round(3).values,
        "media_test": Xs_test[num_cols].mean().round(3).values,
    },
    index=num_cols,
)
print(summary.to_string())

# En train la media es ~0 y la desviación ~1 (por construcción); en test solo aproximadamente,
# porque se usan μ y σ de train.
assert np.allclose(Xs_train[num_cols].mean(), 0, atol=1e-6)
assert np.allclose(Xs_train[num_cols].std(ddof=0), 1, atol=1e-6)

                   rango_original  media_train  std_train  media_test
age                        5 – 95          0.0        1.0       0.016
time_in_hospital           1 – 14         -0.0        1.0      -0.003
num_lab_procedures        1 – 132         -0.0        1.0      -0.013
num_procedures              0 – 6          0.0        1.0       0.014
num_medications            1 – 81          0.0        1.0      -0.008
number_outpatient          0 – 42         -0.0        1.0       0.003
number_emergency           0 – 37         -0.0        1.0       0.001
number_inpatient           0 – 12         -0.0        1.0       0.003
number_diagnoses           1 – 16         -0.0        1.0       0.009


## Resumen
- Dataset analítico: **69 973 pacientes únicos**, **8.97 %** positivos.
- Split estratificado 55 978 / 13 995.
- **182 variables** tras el one-hot (`medical_specialty` colapsada a 12 columnas).
- Sin fuga: split antes de imputar; el preprocesador se ajusta con train dentro de un `Pipeline`
  que luego se reutiliza en la validación cruzada (se re-ajusta en cada fold).
- Escalado `StandardScaler` (z-score, ajustado con train) solo para regresión logística y SVM; los árboles no lo necesitan.